In [25]:

from pymongo import MongoClient
import pandas as pd
from pathlib import Path
import base64
from bson import Binary
from bson import ObjectId


In [26]:
def sanitize_df(df: pd.DataFrame) -> pd.DataFrame:
    def convert(x):
        # bytes → base64 string
        if isinstance(x, (bytes, bytearray)):
            return base64.b64encode(x).decode("ascii")

        # ObjectId → string
        if isinstance(x, ObjectId):
            return str(x)

        # list / tuple → xử lý từng phần tử
        if isinstance(x, (list, tuple)):
            return [convert(v) for v in x]

        # dict → xử lý từng value
        if isinstance(x, dict):
            return {k: convert(v) for k, v in x.items()}

        return x

    return df.applymap(convert)

class TrafficMongoLoader:
    def __init__(self, uri="mongodb://localhost:27017/", db_name="bktraffic"):
        self.client = MongoClient(uri)
        self.db = self.client[db_name]

    def load_segments(self):
        col = self.db["Segments"]
        data = list(col.find({}, {"_id": 0}))
        return pd.DataFrame(data)

    def load_segment_reports(self, limit=None):
        col = self.db["SegmentReports"]
        cursor = col.find({}, {"_id": 0})
        if limit:
            cursor = cursor.limit(limit)
        return pd.DataFrame(list(cursor))

    def load_segment_status(self):
        col = self.db["SegmentStatus"]
        data = list(col.find({}, {"_id": 0}))
        return pd.DataFrame(data)

    def load_nodes(self):
        col = self.db["Nodes"]
        data = list(col.find({}, {"_id": 0}))
        return pd.DataFrame(data)

    def load_nodesDirect(self):
        col = self.db["NodesDirect"]
        data = list(col.find({}, {"_id": 0}))
        return pd.DataFrame(data)

    def load_way_osm(self, limit=None):
        col = self.db["WayOSM"]
        cursor = col.find({}, {"_id": 0})
        if limit:
            cursor = cursor.limit(limit)
        return pd.DataFrame(list(cursor))

    def load_node_osm(self):
        col = self.db["NodeOSM"]
        data = list(col.find({}, {"_id": 0}))
        return pd.DataFrame(data)

    def load_weather(self):
        col = self.db["WeatherInfo"]
        data = list(col.find({}, {"_id": 0}))
        return pd.DataFrame(data)

    def load_references(self):
        col = self.db["References"]
        data = list(col.find({}, {"_id": 0}))
        return pd.DataFrame(data)

    def load_street(self):
        col = self.db["Streets"]
        data = list(col.find({}, {"_id": 0}))
        return pd.DataFrame(data)

    def load_traffic_report(self):
        col = self.db["traffic_report"]
        data = list(col.find({}, {"_id": 0}))
        return pd.DataFrame(data)

    def load_traffic_status(self):
        col = self.db["traffic_status"]
        data = list(col.find({}, {"_id": 0}))
        return pd.DataFrame(data)

    def load_path_histories(self, limit=None):
        col = self.db["PathHistories"]
        cursor = col.find({}, {"_id": 0})
        if limit:
            cursor = cursor.limit(limit)
        return pd.DataFrame(list(cursor))



In [27]:
loader = TrafficMongoLoader()

# 1) Lấy đúng đường dẫn PROJECT
PROJECT = Path.cwd().parents[2]          # Urban-Traffic-Links
RAW_DIR = PROJECT / "data" / "raw" / "bktraffic"
JSON_DIR = PROJECT / "data" / "raw" / "bktraffic" / 'json'

RAW_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)

print("Saving to:", RAW_DIR)

# 2) Dictionary: collection_name → dataframe
collections = {
    "Segments":        loader.load_segments(),
    "SegmentStatus":   loader.load_segment_status(),
    "SegmentReports":  loader.load_segment_reports(),
    "Nodes":           loader.load_nodes(),
    "NodesDirect":     loader.load_nodesDirect(),
    "WayOSM":          loader.load_way_osm(),
    "NodeOSM":         loader.load_node_osm(),
    "WeatherInfo":     loader.load_weather(),
    "References":      loader.load_references(),
    "Streets":         loader.load_street(),
    "traffic_report":  loader.load_traffic_report(),
    "traffic_status":  loader.load_traffic_status(),
    "PathHistories":   loader.load_path_histories()
}


# 3) Lưu sang JSON hoặc parquet
for name, df in collections.items():
    if df is None or df.empty:
        print(f"[SKIP] {name} (empty)")
        continue

    # Bỏ cột images nếu không cần
    if name == "SegmentReports" and "images" in df.columns:
        df = df.drop(columns=["images"])

    df_clean = sanitize_df(df)

    json_path = JSON_DIR / f"{name}.json"
    try:
        df_clean.to_json(json_path, orient="records",
                         force_ascii=False, indent=2)
        print(f"[OK JSON] {name} → {json_path.name}")
    except Exception as e:
        print(f"[SKIP JSON] {name} vì lỗi: {e}")




Saving to: C:\AI\Specialized_Project2_github\Urban-Traffic-Links\data\raw\bktraffic


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17360\2376966921.py:21: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(convert)


[OK JSON] Segments → Segments.json
[SKIP] SegmentStatus (empty)


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17360\2376966921.py:21: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(convert)


[OK JSON] SegmentReports → SegmentReports.json


...

[OK JSON] Nodes → Nodes.json
[OK JSON] NodesDirect → NodesDirect.json


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17360\2376966921.py:21: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(convert)


[OK JSON] WayOSM → WayOSM.json
[OK JSON] NodeOSM → NodeOSM.json
[OK JSON] WeatherInfo → WeatherInfo.json
[OK JSON] References → References.json
[OK JSON] Streets → Streets.json


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17360\2376966921.py:21: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(convert)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17360\2376966921.py:21: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(convert)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17360\2376966921.py:21: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(convert)


[OK JSON] traffic_report → traffic_report.json
[OK JSON] traffic_status → traffic_status.json
[OK JSON] PathHistories → PathHistories.json
